In [1]:
!py --version

Python 3.10.8


# import lib

In [2]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch 
from paddleocr import PaddleOCR
import paddle
import pprint
import numpy as np
from PIL import Image

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://gith

# input

In [3]:
image_path = "./training data/1.jpg"

# function

#### get min and max points 

In [4]:
# to get the max and min points 
def getMinMaxPoint(poly):
    x0 = int(poly[:,0].min())
    y0 = int(poly[:,1].min())
    x1 = int(poly[:,0].max())
    y1 = int(poly[:,1].max())
    return [x0, y0, x1, y1] 

#### tokenize the sentence and box 

In [5]:
def tokenize_sent(xy_points, sentence_split, sentence):
    print(len(sentence))
    letter_width=(xy_points[2]-xy_points[0])/len(sentence)
    x = xy_points[0]
    y0, y1 = xy_points[1], xy_points[3]

    for i, word in enumerate(sentence_split):
        word_width = len(word) * letter_width
        x_end = x + word_width

        # print([int(x), y0, int(x_end), y1])
        token_word.append(word)
        token_poly.append([int(x), y0, int(x_end), y1])
        x = x_end

        # add space after word (except last)
        if i < len(sentence_split) - 1:
            x += letter_width

#### normalize points

In [6]:
def normalize(token_ploy, width, height):
    for box in token_ploy:
        boxes.append([
            int(1000 * box[0] / width),
            int(1000 * box[1] / height),
            int(1000 * box[2] / width),
            int(1000 * box[3] / height),
            ])

# define list 

In [7]:
token_word=[]
token_poly=[]
boxes=[]

# move the model to gpu

In [8]:
gpu_or_cpu="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(gpu_or_cpu) #used to set the model to run in cpu or cuda 

print("Using device:", device)

model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base")
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base",apply_ocr=False)

# Move model to GPU
model.to(device)
device

Using device: cpu


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 212/212 [00:00<00:00, 15448.37it/s]
LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor,

device(type='cpu')

# run ocr on gpu

In [9]:
print(paddle.device.get_device())
ocr = PaddleOCR(use_textline_orientation=True, lang='en', device='gpu')
result = ocr.predict(image_path)

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


gpu:0


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


# get x and y points from the ocr result
# send the ocr result poly into getMinMaxPoint()
# get sentence from the ocr result

In [10]:
# poly=result[0]["rec_polys"][0][:]
# xy_points=getMinMaxPoint(poly)
# sentence= result[0]["rec_texts"][0]
# sentence_split = result[0]["rec_texts"][0].split()

# print(len(result[0]["rec_polys"]))
# print(len(result[0]["rec_texts"]))

# convert the ocr result to tokenize

In [11]:
for i,(box,word) in enumerate(zip(result[0]["rec_polys"],result[0]["rec_texts"])):
    if len(word) == 0:
        continue  
    points_max_min=getMinMaxPoint(box[:])
    tokenize_sent(points_max_min,word.split(),word)
    # print(points_max_min,tokenize_sent)

22
11
14
5
45
13
9
13
21
15
28
26
21
16
32
15
17
5
7
29
15
18
26
6
18
11
10
22
17
2
20
7
3
8
4
4
3
6
3
4
2
14
1
31
8
4
6
8
8
3
9
1
24
8
4
7
5
5
3
8
1
18
8
4
5
5
5
3
6
1
4
10
4
4
7
5
5
3
8
1
3
11
6
4
7
5
5
3
8
9
9
1
1
6
9
1
6
2
9
8
9
8
5
9
7
5
6
8
5
9
28
8
43
7
4
10
5
5
4
6
4
6
10
9
2
8
2
8
9
8
2
6
2
6
8
6
9
8
8
9
11
26
64
56
18
36


# import image 
# get image size 

In [12]:
image = Image.open(image_path).convert("RGB")
width, height = image.size

# normalize the box

In [13]:
normalize(token_poly,width,height)

# layoutlvm3 procesing 

#### convert the string to number where neural network can understand

In [14]:
encoding = processor(
    images=image,
    text=token_word,
    boxes=boxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True
)

#### move the model to same device (cpu or gpu)

In [15]:
for k, v in encoding.items():
    encoding[k] = v.to(device)

#### get prediction
#### 
####

In [16]:
# Get predictions
outputs = model(**encoding)

# The model outputs logits for each token
logits = outputs.logits  # shape: [batch_size, seq_len, num_labels]

# Get predicted class for each token
predictions = torch.argmax(logits, dim=-1)

####
####
####

In [17]:
# Suppose your model has these label mappings
labels = model.config.id2label

predicted_labels = [labels[p.item()] for p in predictions[0][:len(token_word)]]

for token_word, label in zip(token_word, predicted_labels):
    print(token_word, "->", label)

PARMESHWARA -> LABEL_1
STATIONERY -> LABEL_0
Involce -> LABEL_0
No. -> LABEL_0
e-Way -> LABEL_1
Bill -> LABEL_1
No. -> LABEL_0
Dated -> LABEL_0
Near -> LABEL_1
Hasanamba -> LABEL_1
Temple -> LABEL_1
Circle, -> LABEL_1
Balegara -> LABEL_1
Street -> LABEL_1
PS/25-26/1712 -> LABEL_1
18-Sep-26 -> LABEL_1
Delivery -> LABEL_1
Note -> LABEL_1
Mode/Terms -> LABEL_1
of -> LABEL_1
Payment -> LABEL_1
Hassan -> LABEL_0
- -> LABEL_1
573201 -> LABEL_1
Mob -> LABEL_1
: -> LABEL_1
9448919881, -> LABEL_1
7760638884 -> LABEL_1
GSTIN/UIN: -> LABEL_1
29AFHPM3915J1Z5 -> LABEL_1
Reference -> LABEL_1
No. -> LABEL_1
& -> LABEL_1
Date. -> LABEL_1
Other -> LABEL_1
References -> LABEL_1
State -> LABEL_1
Name -> LABEL_1
: -> LABEL_1
Kamataka, -> LABEL_1
Code -> LABEL_1
: -> LABEL_1
29 -> LABEL_1
Buyer -> LABEL_1
(Bill -> LABEL_1
to) -> LABEL_1
Buyer's -> LABEL_1
Order -> LABEL_1
No. -> LABEL_1
Dated -> LABEL_0
MCE -> LABEL_1
CES -> LABEL_0
Malnad -> LABEL_0
College -> LABEL_1
of -> LABEL_0
Engineering -> LABEL_0
